In [2]:
import requests
import pandas as pd
from datetime import datetime, date, timezone, timedelta
import time
import random
from typing import Any
import json
from pathlib import Path

from renewables_permitting.utils import as_list

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# Funciones API BOE bronze

- Cada día guardo la respuesta completa obtenida de la API del BOE para esa fecha. Guardar en S3 bronze.
- Controlar días sin BOE o errores 404.
- Concatenar los DataFrame diarios y guardar el resultado en CSV/Parquet.

In [3]:
def get_sumario_boe(fecha: str) -> dict[str, Any]:
    """
    Obtiene el sumario del BOE correspondiente a una fecha determinada.

    Realiza una petición HTTP GET a la API de datos abiertos del BOE y
    devuelve la respuesta en formato JSON ya deserializada como un
    diccionario de Python.

    La fecha debe especificarse en formato AAAAMMDD. Por ejemplo:
    - 20230102 → BOE de 2 de enero de 2023
    - 20240529 → BOE de 29 de mayo de 2024

    Parámetros
    ----------
    fecha : str
        Fecha de publicación del BOE en formato AAAAMMDD.

    Retorna
    -------
    dict
        Respuesta JSON devuelta por la API del BOE.

    Raises
    ------
    requests.exceptions.HTTPError
        Si la API devuelve un código de error HTTP
        (por ejemplo, 404 si no existe el sumario solicitado).

    requests.exceptions.RequestException
        Si ocurre cualquier problema de comunicación con el servidor
        (timeout, error de conexión, etc.).

    Ejemplos
    --------
    >>> data = get_sumario_boe("20230102")
    >>> data["status"]["code"]
    "200"
    """
    url = f"{BASE_URL}/{fecha}"

    response = requests.get(
        url,
        headers={"Accept": "application/json"},
        timeout=30,
    )

    response.raise_for_status()

    return response.json()

In [4]:
def save_json_local(data: dict[str, Any], path: Path) -> None:
    """
    Guarda un diccionario Python como archivo JSON local.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

In [5]:
def count_items_sumario_boe(data: dict[str, Any]) -> int:
    """
    Cuenta de forma aproximada los ítems publicados en un sumario BOE.
    """
    count = 0

    sumario = data.get("data", {}).get("sumario", {})
    diarios = as_list(sumario.get("diario"))

    for diario in diarios:
        for seccion in as_list(diario.get("seccion")):
            for departamento in as_list(seccion.get("departamento")):

                for epigrafe in as_list(departamento.get("epigrafe")):
                    count += len(as_list(epigrafe.get("item")))

                texto = departamento.get("texto")
                if isinstance(texto, dict):
                    count += len(as_list(texto.get("item")))

                count += len(as_list(departamento.get("item")))

    return count

In [6]:
def ingest_sumario_boe_local(
    fecha: str,
    base_dir: Path = Path(BRONZE_DIR / "boe"),
) -> dict[str, Any]:
    """
    Descarga el sumario diario del BOE y lo guarda en local.
    """
    day_dir = base_dir / fecha

    metadata: dict[str, Any] = {
        "date": fecha,
        "source": "BOE API sumario",
        "ingested_at": datetime.now(timezone.utc).isoformat(),
        "status": None,
        "http_status": None,
        "boe_status_code": None,
        "records_downloaded": 0,
        "error_type": None,
        "error": None,
    }

    try:
        data = get_sumario_boe(fecha)

        boe_status_code = data.get("status", {}).get("code")
        metadata["boe_status_code"] = boe_status_code
        metadata["http_status"] = 200

        if boe_status_code != "200":
            metadata["status"] = "failed"
            metadata["error_type"] = "BOE_STATUS_ERROR"
            metadata["error"] = data.get("status", {}).get("text")
            save_json_local(metadata, day_dir / "metadata.json")
            return metadata

        save_json_local(data, day_dir / "sumario.json")

        metadata["status"] = "success"
        metadata["records_downloaded"] = count_items_sumario_boe(data)

        save_json_local(metadata, day_dir / "metadata.json")
        return metadata

    except requests.exceptions.HTTPError as exc:
        response = exc.response
        http_status = response.status_code if response is not None else None

        metadata["http_status"] = http_status

        if http_status == 404:
            metadata["status"] = "no_publication"
            metadata["error_type"] = "HTTP_404"
            metadata["error"] = "No BOE publication for this date"
        else:
            metadata["status"] = "failed"
            metadata["error_type"] = "HTTP_ERROR"
            metadata["error"] = str(exc)

        save_json_local(metadata, day_dir / "metadata.json")
        return metadata

    except requests.exceptions.RequestException as exc:
        metadata["status"] = "failed"
        metadata["error_type"] = "REQUEST_ERROR"
        metadata["error"] = str(exc)
        save_json_local(metadata, day_dir / "metadata.json")
        return metadata

    except ValueError as exc:
        metadata["status"] = "failed"
        metadata["error_type"] = "INVALID_JSON"
        metadata["error"] = str(exc)
        save_json_local(metadata, day_dir / "metadata.json")
        return metadata

In [7]:
# TODO: crear función ingest_sumario_boe_s3 para producción.
def ingest_sumario_boe_s3 (fecha: str):
    pass

# Descarga en batch de sumarios BOE 

In [ ]:
def iter_dates(start: str, end: str):
    """
    Genera fechas en formato AAAAMMDD entre start y end, ambos incluidos.
    """
    start_date = date.fromisoformat(start)
    end_date = date.fromisoformat(end)

    current = start_date
    while current <= end_date:
        yield current.strftime("%Y%m%d")
        current += timedelta(days=1)

In [11]:
#fechas = list(iter_dates("2026-06-10", date.today().isoformat()))
#fechas = list(iter_dates("2023-01-01", "2023-12-31"))
fechas = ["20230428"]

for fecha in fechas:
    ingest_sumario_boe_local(fecha)

    time.sleep(random.uniform(0.2, 1.5))